In [ ]:
!pip install yfinance -i https://pypi.tuna.tsinghua.edu.cn/simple

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import akshare as ak  # 昨晚实测最稳的数据源

C:\Users\DELL\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
# 1. 特征工程函数
def prepare_data(df):
    df = df.copy()
    # MA5
    df['MA5'] = df['close'].rolling(window=5).mean()
    # MACD
    ema12 = df['close'].ewm(span=12, adjust=False).mean()
    ema26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd_line'] = ema12 - ema26
    df['signal_line'] = df['macd_line'].ewm(span=9, adjust=False).mean()
    # RSI
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-9)
    df['rsi_14'] = 100 - (100 / (1 + rs))
    # Target: 下一日收益率
    df['target'] = df['close'].shift(-1) / df['close'] - 1
    df.dropna(inplace=True)
    return df

# 2. 数据集封装类
class StockDataset(Dataset):
    def __init__(self, data, features, seq_len=30):
        self.X = data[features].values.astype(np.float32)
        self.y = data['target'].values.astype(np.float32)
        self.seq_len = seq_len
    def __len__(self):
        return len(self.X) - self.seq_len
    def __getitem__(self, idx):
        return (torch.tensor(self.X[idx : idx + self.seq_len]), 
                torch.tensor(self.y[idx + self.seq_len]))

# 3. LSTM 模型类
class QuantLSTM(nn.Module):
    def __init__(self, input_sz, hidden_sz, num_layers):
        super(QuantLSTM, self).__init__()
        self.lstm = nn.LSTM(input_sz, hidden_sz, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_sz, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

In [4]:
# 抓取沪深300
df_raw = ak.stock_zh_index_daily(symbol="sh000300")
df_raw.columns = ['date', 'open', 'close', 'high', 'low', 'volume']
df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw = df_raw[df_raw['date'] >= '2020-01-01'].reset_index(drop=True)

# 运行刚才定义的 prepare_data
df_proc = prepare_data(df_raw)

# 核心：归一化
FEATURE_COLS = ['close', 'volume', 'MA5', 'macd_line', 'signal_line', 'rsi_14']
scaler = StandardScaler()
df_proc[FEATURE_COLS] = scaler.fit_transform(df_proc[FEATURE_COLS])

In [5]:
# --- 1. 按时间切分 ---滚动回测的分水岭
train_df = df_proc[df_proc['date'] < '2025-01-01'].copy()
test_df = df_proc[df_proc['date'] >= '2025-01-01'].copy()

# --- 2. 实例化 Dataset ---
train_dataset = StockDataset(train_df, FEATURE_COLS)
test_dataset = StockDataset(test_df, FEATURE_COLS)

# --- 3. 封装 DataLoader ---
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"📊 训练集天数: {len(train_df)} | 测试集天数: {len(test_df)}")

📊 训练集天数: 1199 | 测试集天数: 301


In [6]:
# 1. 初始化模型
model = QuantLSTM(len(FEATURE_COLS), 64, 2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# 2. 训练循环
model.train()
print("🚀 正在复刻 0.0715 的训练过程...")
for epoch in range(30):
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/30, Avg Loss: {total_loss/len(train_loader):.6f}")

print("✅ 训练完成，模型已就绪！")

🚀 正在复刻 0.0715 的训练过程...
Epoch 10/30, Avg Loss: 0.000120
Epoch 20/30, Avg Loss: 0.000121
Epoch 30/30, Avg Loss: 0.000119
✅ 训练完成，模型已就绪！


In [7]:
model.eval()
test_preds = []
test_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        test_preds.extend(outputs.squeeze().numpy())
        test_labels.extend(labels.numpy())

# 计算复现的 Rank IC
final_rank_ic = pd.Series(test_preds).rank().corr(pd.Series(test_labels).rank())

print("\n" + "="*40)
print(f"✨ 复现成功！2026年样本外 Rank IC: {final_rank_ic:.4f}")
print(f"📊 预测值标准差: {np.std(test_preds):.6f}")
print("="*40)


✨ 复现成功！2026年样本外 Rank IC: 0.0801
📊 预测值标准差: 0.002782


In [8]:
# 在 Notebook 底部跑这一行，把处理好的数据存下来，省得下次又要重抓
df_proc.to_csv('HS300_Processed_2026.csv', index=False)